In [7]:
from preprocessing_EEG import extract_labeled_events, preprocessing_grandchamp

bids_root = "C:\\Users\\aadel\\Desktop\\GCID\\Cuarto\\Segundo Cuatrimestre\\TFG\\data\\ds001787-download"

stage = 6

if stage == 3:
    epochs, ica = preprocessing_grandchamp(
        sub=1, stage=3, session=1, bids_root=bids_root, fast_mode=True
    )
else:
    epochs = preprocessing_grandchamp(
        sub=1, stage=stage, session=1, bids_root=bids_root, fast_mode=True
    )


=== Stage 6: Final Epoch Rejection ===
Reading C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing\001_01_epochs_ica_a-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =  -10000.00 ...       0.00 ms
        0 CTF compensation matrices available
Adding metadata with 14 columns
25 matching events found
No baseline correction applied
0 projection items activated
Loaded 25 epochs

Rejecting manually marked bad epochs...

✓ Rejection complete:
  Initial epochs: 25
  Rejected: 0 (0.0%)
  Final epochs: 25

  Label distribution:
    On-Task: 17 → 17
    Mind Wandering: 8 → 8

✓ Final epochs saved: 001_01_epochs_ica_a2-epo.fif

✓ Summary saved: preprocessing\001_01_final_summary.txt

🎉 PREPROCESSING PIPELINE COMPLETE!

Your data is ready for analysis!

Final cleaned epochs: preprocessing\001_01_epochs_ica_a2-epo.fif
Load with: epochs = mne.read_epochs('preprocessing\001_01_epochs_ica_a2-epo.fif')


In [14]:
from components_extraction import extract_features_from_epochs
import mne

path_input = 'C:\\Users\\aadel\\Desktop\\GCID\\Cuarto\\Segundo Cuatrimestre\\TFG\\python\\preprocessing\\001_01_epochs_ica_a2-epo.fif'
epochs = mne.read_epochs(path_input, preload=True)

# 2. Extraer características
# Nota: El script usará PO7, Pz, PO8 y Fz (mapeados a A10, A19, B7, C21)
df = extract_features_from_epochs(
    epochs,
    baseline_window=(-10, -9), # Primer segundo de la época
    signal_window=(-1, 0)      # Último segundo antes del aviso
)

# 3. Guardar resultados
df.to_csv('features_MW_study.csv', index=False)
print("\n" + "="*30)
print(f"¡ÉXITO! Archivo guardado como 'features_MW_study.csv'")
print(f"Dimensiones de la matriz: {df.shape}")

# 4. Análisis rápido de control (¿Funciona nuestra hipótesis?)
print("\nPROMEDIOS POR ESTADO (ALPHA EN Pz):")
resumen = df.groupby('label')['alpha_power_Pz_signal'].mean()
print(resumen)
print("="*30)

Reading C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing\001_01_epochs_ica_a2-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =  -10000.00 ...       0.00 ms
        0 CTF compensation matrices available
Adding metadata with 14 columns
25 matching events found
No baseline correction applied
0 projection items activated

FEATURE EXTRACTION
Epochs: 25
Channels: ['PO7', 'Pz', 'PO8', 'Fz']
Alpha band: 8.5-12 Hz
Theta band: 4-8 Hz
Baseline window: -10 to -9 s
Signal window: -1 to 0 s
⚠️  Warning: Channel Fz/C21 not found

✓ Using channels: {'PO7': 'A10', 'Pz': 'A19', 'PO8': 'B7'}
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).

Baseline samples: 257
Signal samples: 257

--- Creating Filters ---
⚠️  Warning: Filter SSE = 39.1677 (should be < 1.0)
   Consider increasing filter length or adjusting parameters
⚠️  Warning: Filter SSE = 28.6530 (should be < 1.0)
   Consider increasing filter length or adjusting

In [10]:
path_input = 'C:\\Users\\aadel\\Desktop\\GCID\\Cuarto\\Segundo Cuatrimestre\\TFG\\python\\preprocessing\\001_01_epochs_ica_a2-epo.fif'
epochs = mne.read_epochs(path_input, preload=True)
print(epochs.ch_names)

Reading C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing\001_01_epochs_ica_a2-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =  -10000.00 ...       0.00 ms
        0 CTF compensation matrices available
Adding metadata with 14 columns
25 matching events found
No baseline correction applied
0 projection items activated
['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18', 'B19', 'B20', 'B21', 'B22', 'B23', 'B24', 'B25', 'B26', 'B27', 'B28', 'B29', 'B30', 'B31', 'B32', 'EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'GSR1', 'GSR2', 'Erg1', 'Erg2', 'Resp', 'Plet', 'Temp', 'Status']


In [2]:
import numpy as np
from components_extraction import compute_ispc
# 1. Configuración básica
sfreq = 256                  # Frecuencia de muestreo (Hz)
t = np.arange(0, 1, 1/sfreq) # Vector de tiempo de 1 segundo
freq = 10                    # Frecuencia de la señal (10 Hz - Alpha)

# 2. Creamos la señal X (Referencia)
# Una señal analítica es exp(i * 2 * pi * f * t)
analytic_x = np.exp(1j * 2 * np.pi * freq * t)

# 3. Creamos la señal Y (Con desfase variable)
# Le añadimos un desfase inicial (pi/4) y un poco de ruido aleatorio
# para que la sincronía no sea perfecta (ISPC < 1.0)
ruido_fase = np.random.normal(0, 0.5, size=t.shape) # Ruido de fase
analytic_y = np.exp(1j * (2 * np.pi * freq * t + np.pi/4 + ruido_fase))

result = compute_ispc(analytic_x, analytic_y)
print(result)

0.8827537978067765
